# Model Evaluation & Cross-Validation

Before you trust *any* model's reported performance — including the ones in the
[Regression](../02-regression/linear-regression.ipynb) and
[Trees](../04-trees/decision-trees.ipynb) chapters — you need a principled way to
estimate how it will do on **unseen** data. That's what train/test splitting and
cross-validation give you.

We use [`smartcore`](https://docs.rs/smartcore), which (unlike `linfa`) bundles
`model_selection` and `metrics` utilities and a few ready-to-use datasets. Its
matrix type is `DenseMatrix` rather than `ndarray`.

```{note}
This chapter uses `smartcore`'s built-in **breast cancer** dataset (569 samples,
30 features, binary target) so we're learning one new idea — evaluation — on
familiar-shaped tabular data, not a new dataset *and* a new concept at once.
```

In [ ]:
:dep smartcore = { version = "0.3", features = ["datasets"] }
:dep ndarray = { version = "0.16" }
:dep model-selection-rs = { version = "0.1.0" }
use smartcore::linalg::basic::matrix::DenseMatrix;
use smartcore::linalg::basic::arrays::Array;
use smartcore::dataset::breast_cancer;

// Load the dataset. We build `x` and `y` inside a block so the intermediate
// `Dataset` value (whose type evcxr can't name to persist) stays local; the
// explicitly-typed pair is what survives to later cells.
let (x, y): (DenseMatrix<f32>, Vec<i32>) = {
    let ds = breast_cancer::load_dataset();
    let x = DenseMatrix::new(ds.num_samples, ds.num_features, ds.data.clone(), false);
    let y = ds.target.iter().map(|&v| v as i32).collect();
    (x, y)
};
println!("features: {} samples x {} features", x.shape().0, x.shape().1);
println!("labels: {} (classes 0/1)", y.len());

## Train/test split

The simplest honest estimate: hold out a slice of data the model never sees
during training, then score on it. `train_test_split` shuffles and splits both
`x` and `y` together (30% test here, fixed seed for reproducibility).

In [ ]:
use smartcore::model_selection::train_test_split;
use smartcore::metrics::accuracy;
use smartcore::linear::logistic_regression::LogisticRegression;

{
    let (xtr, xte, ytr, yte) = train_test_split(&x, &y, 0.3, true, Some(42));
    let model = LogisticRegression::fit(&xtr, &ytr, Default::default()).unwrap();
    let pred = model.predict(&xte).unwrap();
    println!("train rows = {}, test rows = {}", xtr.shape().0, xte.shape().0);
    println!("hold-out accuracy = {:.3}", accuracy(&yte, &pred));
}

## K-fold cross-validation

A single split is noisy — you got *one* lucky or unlucky test set. **K-fold**
cross-validation splits the data into *k* folds, trains on *k−1* and tests on the
held-out fold, rotating through all *k*. Averaging the per-fold scores gives a
far more stable estimate.

In [ ]:
use smartcore::model_selection::{cross_validate, KFold};
use smartcore::api::SupervisedEstimator;  // brings `LogisticRegression::new()` into scope

{
    let cv = KFold::default().with_n_splits(5);
    let results = cross_validate(
        LogisticRegression::new(),
        &x, &y,
        Default::default(),
        &cv,
        &accuracy,
    ).unwrap();
    // `test_score` is a public field (per-fold scores); the mean is a method.
    println!("per-fold test accuracy: {:?}", results.test_score);
    println!("mean CV accuracy = {:.3}", results.mean_test_score());
}

## Stratified K-fold — preserving class balance

Plain K-fold shuffles and cuts blindly, so with **imbalanced classes** a fold's
class ratio can drift from the whole dataset's — and in the worst case a rare
class is missing from a fold entirely, poisoning that fold's score. **Stratified**
K-fold keeps each fold's class proportions close to the full data's.

`smartcore` 0.3 has no `StratifiedKFold`, but
[`model-selection-rs`](https://crates.io/crates/model-selection-rs) does — along
with group-aware, time-series, and repeated splitters. Its splitters return fold
**indices** `(train, test)`, so they're model- and matrix-agnostic: you get the
folds, then slice whatever data structure you already have. Below, the same data
under plain `KFold` vs `StratifiedKFold` — watch the per-fold class-1 fraction:

In [ ]:
// `KFold` here would clash with smartcore's own `KFold` (imported above), so we
// alias model-selection-rs's as `MsKFold`.
use model_selection_rs::splitters::{CvSplitter, KFold as MsKFold, StratifiedKFold};
use ndarray::Array1;

{
    let n = y.len();
    let y_arr = Array1::from(y.clone());                 // labels as ndarray for the stratifier
    let frac1 = |idx: &[usize]| idx.iter().filter(|&&i| y[i] == 1).count() as f64 / idx.len() as f64;
    let overall = y.iter().filter(|&&v| v == 1).count() as f64 / n as f64;
    println!("overall class-1 fraction = {:.3}\n", overall);

    println!("plain KFold(5)      — test-fold class-1 fraction drifts:");
    for (f, (_tr, te)) in MsKFold::new(5).unwrap().with_shuffle(0).split(n).unwrap().iter().enumerate() {
        println!("  fold {f}: {:.3}", frac1(te));
    }

    println!("StratifiedKFold(5)  — each fold held close to {:.3}:", overall);
    for (f, (_tr, te)) in StratifiedKFold::new(5, &y_arr).unwrap().split(n).unwrap().iter().enumerate() {
        println!("  fold {f}: {:.3}", frac1(te));
    }
}

The mean across folds is the number to report — notice it differs from the
single hold-out score above, which is exactly why we prefer it.

## Metrics beyond accuracy

Accuracy hides a lot, especially with [imbalanced classes](../01b-eda/exploratory-data-analysis.ipynb).
`smartcore::metrics` covers the usual classification and regression measures:

In [ ]:
use smartcore::metrics::{precision, recall, f1, mean_squared_error, r2};

{
    let (xtr, xte, ytr, yte) = train_test_split(&x, &y, 0.3, true, Some(42));
    let model = LogisticRegression::fit(&xtr, &ytr, Default::default()).unwrap();
    let pred = model.predict(&xte).unwrap();
    // Classification metrics (task: is the tumour malignant?)
    println!("accuracy  = {:.3}", accuracy(&yte, &pred));
    // precision/recall/f1 expect float-typed labels, so cast the 0/1 labels:
    let yte_f: Vec<f32> = yte.iter().map(|&v| v as f32).collect();
    let pred_f: Vec<f32> = pred.iter().map(|&v| v as f32).collect();
    println!("precision = {:.3}", precision(&yte_f, &pred_f));
    println!("recall    = {:.3}", recall(&yte_f, &pred_f));
    println!("f1        = {:.3}", f1(&yte_f, &pred_f, 1.0));
    // Regression metrics, illustrated on plain numeric vectors:
    let truth = vec![3.0_f32, 5.0, 2.5, 7.0];
    let est   = vec![2.8_f32, 5.2, 2.7, 6.6];
    println!("MSE = {:.3}", mean_squared_error(&truth, &est));
    println!("R2  = {:.3}", r2(&truth, &est));
}

## Pitfalls & gaps

- **Data leakage.** Fit scalers/imputers (from the [ETL chapter](../01c-etl/data-preparation.ipynb))
  on the *training* fold only, then apply them to the test fold. Fitting them on
  the full dataset before splitting leaks test information into training and
  inflates your scores.
- **Stratified splits.** With imbalanced classes you want each fold to preserve
  the class ratio. `smartcore` 0.3 has no `StratifiedKFold`, so we used
  [`model-selection-rs`](https://crates.io/crates/model-selection-rs) above — it
  also provides group-aware (`GroupKFold`), time-series (`TimeSeriesSplit`), and
  repeated splitters, all returning fold indices you can apply to any model.
- **The same idea, automated.** The [AutoML chapter](../06-automl/automl-classification.ipynb)
  runs this exact cross-validation loop internally to compare models — it's not
  magic, just this chapter applied across a model zoo.

Next: back to modelling with [linear regression](../02-regression/linear-regression.ipynb) —
now you can evaluate it properly.